# OKX加密货币数据收集

本Notebook用于从OKX API获取BTC/USDT的15分钟K线数据，并计算技术指标。

**作者**: guozhicheng  
**学号**: 1305996

## 单元格1：安装依赖（如果之前没装全）

In [ ]:
# 安装所需依赖包（已安装可跳过）
!pip install ccxt pandas ta python-dotenv

## 单元格2：导入所需库

In [ ]:
import ccxt
import pandas as pd
import ta
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

## 单元格3：加载API凭证

In [ ]:
# 加载.env文件中的API信息
load_dotenv()

api_key = os.getenv("OKX_API_KEY")
api_secret = os.getenv("OKX_API_SECRET")
passphrase = os.getenv("OKX_PASSPHRASE")
demo_mode = os.getenv("OKX_DEMO_MODE", "True").lower() == "true"

print(f"API Key: {api_key[:10]}..." if api_key else "API Key not found")
print(f"Demo Mode: {demo_mode}")

## 单元格4：初始化OKX客户端

In [ ]:
# 连接OKX API
exchange = ccxt.okx({
    'apiKey': api_key,
    'secret': api_secret,
    'password': passphrase,
    'enableRateLimit': True,  # 自动遵守API速率限制
    'options': {
        'defaultType': 'spot',  # 现货交易，期货改为'swap'
        'demo': demo_mode
    }
})

print(f"成功连接到OKX {'模拟' if demo_mode else '实盘'} API")

## 单元格5：定义数据获取函数

In [ ]:
def fetch_ohlcv_data(symbol, timeframe, start_date, end_date):
    """
    从OKX API获取历史K线数据
    :param symbol: 交易对，如'BTC/USDT'
    :param timeframe: K线周期，如'15m'
    :param start_date: 开始日期，datetime对象
    :param end_date: 结束日期，datetime对象
    :return: 包含OHLCV数据的DataFrame
    """
    all_candles = []
    current_time = int(start_date.timestamp() * 1000)
    end_time = int(end_date.timestamp() * 1000)
    
    while current_time < end_time:
        try:
            # 每次最多获取100根K线（OKX API限制）
            candles = exchange.fetch_ohlcv(
                symbol=symbol,
                timeframe=timeframe,
                since=current_time,
                limit=100
            )
            
            if not candles:
                break
                
            all_candles.extend(candles)
            current_time = candles[-1][0] + 1  # 移动到下一个时间戳
            
            # 小延迟避免触发速率限制
            time.sleep(0.1)
            
        except Exception as e:
            print(f"获取数据出错: {e}")
            time.sleep(5)
            continue
    
    # 转换为DataFrame
    df = pd.DataFrame(
        all_candles,
        columns=['timestamp', 'open', 'high', 'low', 'close', 'volume']
    )
    
    # 转换时间戳为北京时间（UTC+8）
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms') + timedelta(hours=8)
    
    # 去除重复行
    df = df.drop_duplicates(subset=['timestamp'])
    
    # 过滤到指定日期范围
    df = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]
    
    return df

## 单元格6：获取历史K线数据

In [ ]:
# 定义参数
SYMBOL = 'BTC/USDT'
TIMEFRAME = '15m'
# 收集最近30天的数据（可根据需要调整日期）
START_DATE = datetime(2026, 4, 10, 0, 0, 0)
END_DATE = datetime(2026, 5, 10, 23, 59, 59)

print(f"正在获取 {SYMBOL} {TIMEFRAME} 数据，时间范围：{START_DATE} 至 {END_DATE}...")
df = fetch_ohlcv_data(SYMBOL, TIMEFRAME, START_DATE, END_DATE)
print(f"成功获取 {len(df)} 根K线")

## 单元格7：计算技术指标

In [ ]:
# 计算EMA21和EMA50（趋势指标）
df['ema21'] = ta.trend.ema_indicator(close=df['close'], window=21)
df['ema50'] = ta.trend.ema_indicator(close=df['close'], window=50)

# 计算RSI（动量指标）
df['rsi'] = ta.momentum.rsi(close=df['close'], window=14)

# 计算布林带（波动率指标）
bb = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2)
df['bb_upper'] = bb.bollinger_hband()
df['bb_middle'] = bb.bollinger_mavg()
df['bb_lower'] = bb.bollinger_lband()
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle'] * 100

# 删除包含缺失值的行（前50行因EMA50计算会有缺失）
df = df.dropna()
print(f"计算指标后最终数据集大小：{len(df)} 行")

## 单元格8：保存数据集到CSV

In [ ]:
# 保存到data文件夹
output_path = '../data/btc_usdt_15m_data.csv'
df.to_csv(output_path, index=False)
print(f"数据集已保存到：{output_path}")

## 单元格9：查看数据集预览

In [ ]:
print("\n数据集前5行：")
print(df.head())

print("\n数据集基本信息：")
print(df.info())

print("\n关键变量描述性统计：")
print(df[['close', 'volume', 'rsi', 'bb_width']].describe())

## 数据可视化（可选）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 绘制价格和EMA
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 价格图
axes[0].plot(df['timestamp'], df['close'], label='收盘价', alpha=0.8)
axes[0].plot(df['timestamp'], df['ema21'], label='EMA21', alpha=0.8)
axes[0].plot(df['timestamp'], df['ema50'], label='EMA50', alpha=0.8)
axes[0].fill_between(df['timestamp'], df['bb_upper'], df['bb_lower'], alpha=0.2, label='布林带')
axes[0].set_ylabel('价格 (USDT)')
axes[0].set_title('BTC/USDT 15分钟K线及技术指标')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI图
axes[1].plot(df['timestamp'], df['rsi'], label='RSI', color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='超买(70)')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='超卖(30)')
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 布林带宽度
axes[2].plot(df['timestamp'], df['bb_width'], label='布林带宽度', color='orange')
axes[2].set_ylabel('布林带宽度 (%)')
axes[2].set_xlabel('时间')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# 格式化x轴日期
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
axes[2].xaxis.set_major_locator(mdates.DayLocator(interval=5))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()